# COMP34812 NLI — Input Embedding Module

**Author:** Kaan Berk Temizel  
**Track:** Natural Language Inference (NLI)  
**Solution:** Category B — Deep Learning (non-transformer)

## Overview
This notebook implements the input embedding pipeline for a ResESIM-based NLI model.
Each token is represented by a 1476-dimensional vector formed by concatenating:

| Component | Dim | Type | Reference |
|---|---|---|---|
| GloVe | 300 | Static, frozen | Pennington et al., 2014 |
| ELMo | 1024 | Contextual, frozen | Peters et al., 2018 |
| CharCNN | 100 | Learned | Kim et al., 2016 |
| POS embeddings | 50 | Learned | — |
| Negation flags | 2 | Rule-based | spaCy dep parse |
| **Total** | **1476** | | |

## Notebook Structure
1. Environment Setup
2. Data Loading
3. Tokenisation & Vocabulary
4. GloVe Embeddings
5. Character CNN
6. POS Embeddings
7. Negation Flags
8. ELMo Pre-computation
9. Input Embedding Module
10. Dataset & DataLoaders
11. Save to Drive

## 1. Environment Setup

**Note:** This notebook has been adapted to run on macOS in a local environment.

### Prerequisites
Install required packages:
```bash
pip install spacy psutil pandas numpy torch requests
python -m spacy download en_core_web_sm
```

### ELMo Pre-computation
ELMo (Peters et al., 2018) requires AllenNLP which is incompatible with Python 3.11+.
If you need to pre-compute ELMo embeddings, you'll need to:
1. Create a separate Python 3.10 virtual environment
2. Install AllenNLP and dependencies
3. Run the pre-computation script (see Section 8)

For now, we'll download pre-computed embeddings or skip ELMo temporarily.

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# FILE PATH CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
# Modify these paths to match your environment

from pathlib import Path

# ── Root Directories ──────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()                    # Current working directory
DATA_DIR = PROJECT_ROOT / 'data'                      # Where CSV files are located
BIN_DIR = PROJECT_ROOT / 'bin'               # Where downloaded binaries go
OUTPUT_DIR = PROJECT_ROOT / 'output'         # Where outputs are saved

# ── Data Files (CSV) ──────────────────────────────────────────────────────────
TRAIN_CSV = DATA_DIR / 'train.csv'
DEV_CSV = DATA_DIR / 'dev.csv'
TRIAL_CSV = DATA_DIR / 'NLI_trial.csv'

# ── Binary Directories ────────────────────────────────────────────────────────
GLOVE_DIR = BIN_DIR / 'glove'                # GloVe embeddings
ELMO_DIR = BIN_DIR / 'elmo'                  # ELMo model files

# ── Specific Binary Files ─────────────────────────────────────────────────────
GLOVE_FILE = GLOVE_DIR / 'glove.6B.300d.txt'
ELMO_OPTIONS_FILE = ELMO_DIR / 'elmo_2x4096_512_2048cnn_2xhighway_options.json'
ELMO_WEIGHTS_FILE = ELMO_DIR / 'elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5'

# ── Pre-computed ELMo Embeddings ──────────────────────────────────────────────
ELMO_TRAIN_PREM = OUTPUT_DIR / 'elmo_train_prem.npy'
ELMO_TRAIN_HYP = OUTPUT_DIR / 'elmo_train_hyp.npy'
ELMO_DEV_PREM = OUTPUT_DIR / 'elmo_dev_prem.npy'
ELMO_DEV_HYP = OUTPUT_DIR / 'elmo_dev_hyp.npy'

# ── Saved Artifacts ───────────────────────────────────────────────────────────
VOCAB_GLOVE_SAVE = OUTPUT_DIR / 'vocab_and_glove.pt'
ELMO_ARRAYS_SAVE = OUTPUT_DIR / 'elmo_arrays.npz'

# ── ELMo Pre-computation Script ───────────────────────────────────────────────
ELMO_PRECOMPUTE_SCRIPT = PROJECT_ROOT / 'elmo_precompute.py'

# ── Print Configuration ───────────────────────────────────────────────────────
print("=" * 80)
print("FILE PATH CONFIGURATION")
print("=" * 80)
print(f"Project Root:     {PROJECT_ROOT}")
print(f"Data Directory:   {DATA_DIR}")
print(f"Binary Directory: {BIN_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print()
print("Data Files:")
print(f"  Train CSV:      {TRAIN_CSV}")
print(f"  Dev CSV:        {DEV_CSV}")
print(f"  Trial CSV:      {TRIAL_CSV}")
print()
print("Binary Files:")
print(f"  GloVe:          {GLOVE_FILE}")
print(f"  ELMo Options:   {ELMO_OPTIONS_FILE}")
print(f"  ELMo Weights:   {ELMO_WEIGHTS_FILE}")
print()
print("Output Files:")
print(f"  Vocab & GloVe:  {VOCAB_GLOVE_SAVE}")
print(f"  ELMo Arrays:    {ELMO_ARRAYS_SAVE}")
print("=" * 80)

FILE PATH CONFIGURATION
Project Root:     /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW
Data Directory:   /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/data
Binary Directory: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin
Output Directory: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/output

Data Files:
  Train CSV:      /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/data/train.csv
  Dev CSV:        /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/data/dev.csv
  Trial CSV:      /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/data/NLI_trial.csv

Binary Files:
  GloVe:          /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/glove/glove.6B.300d.txt
  ELMo Options:   /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo/elmo_2x4096_512_2048cnn_2xhighway_options.json
  ELMo Weights:   /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5

Output Files:
  Vocab & GloVe:  /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/outp

## 0. Configuration — File Paths

**Configure all file paths here.** Modify these paths to match your environment.

In [2]:
# ── Main kernel dependencies ──────────────────────────────────────────────────
# Install required packages (run once)
# !pip install spacy psutil pandas numpy torch requests
# !python -m spacy download en_core_web_sm

import sys
import os
from pathlib import Path
import importlib

# Add current directory to path to import elmo package
sys.path.insert(0, str(Path.cwd()))

# from elmo.downloader import download_and_extract_glove_6B, elmo_downloader
import elmo.downloader as elmo_downloader
importlib.reload(elmo_downloader)

# ── Setup directories ──────────────────────────────────────────────────────────
BIN_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# ── GloVe weights (Pennington et al., 2014) ───────────────────────────────────
print("Downloading GloVe embeddings...")
glove_dir = elmo_downloader.download_and_extract_glove_6B(download_folder=str(GLOVE_DIR))
print(f"GloVe downloaded to: {glove_dir}")

# ── ELMo weights (Peters et al., 2018) — original AllenNLP release ───────────
print("\nDownloading ELMo model files...")
elmo_dir = elmo_downloader.elmo_downloader(download_folder=str(ELMO_DIR))
print(f"ELMo downloaded to: {elmo_dir}")

print('\nEnvironment setup complete.')
print(f'Binary files location: {BIN_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

Zip file already exists at /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/glove/glove.6B.zip
Skipping download.
Extracting files to /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/glove
Extraction complete. GloVe files available in /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/glove
GloVe downloaded to: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/glove

ELMo options file already exists at /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo/elmo_2x4096_512_2048cnn_2xhighway_options.json
Skipping download.
ELMo weights file already exists at /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5
Skipping download.
ELMo model files available in /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo
ELMo downloaded to: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo

Environment setup complete.
Binary files location: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin
Output directory: /Users/vpremakant

In [3]:
# ── Main kernel imports ───────────────────────────────────────────────────────
import re
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import spacy
nlp = spacy.load('en_core_web_sm')

# ── Constants ─────────────────────────────────────────────────────────────────
PAD_TOKEN    = '<PAD>'
UNK_TOKEN    = '<UNK>'
PAD_CHAR     = '<PAD>'
UNK_CHAR     = '<UNK>'
PAD_POS      = '<PAD>'
UNK_POS      = '<UNK>'
MAX_LEN      = 64
MAX_WORD_LEN = 20
MIN_FREQ     = 2
EMBED_DIM    = 300
CHAR_EMBED_DIM = 30
CHAR_OUT_DIM   = 100
POS_EMBED_DIM  = 50
NUM_CLASSES    = 2

NEGATION_WORDS = {
    'not', 'no', 'never', 'neither', 'nor', 'nobody', 'nothing',
    'nowhere', 'hardly', 'barely', 'scarcely', "n't", 'cannot'
}

print('Imports and constants OK')
print(f'Using paths from configuration cell above')

Imports and constants OK
Using paths from configuration cell above


## 2. Data Loading

In [4]:
def load_nli_csv(path):
    """Load NLI CSV with columns: premise, hypothesis, label."""
    df = pd.read_csv(Path(path))
    assert {'premise', 'hypothesis', 'label'}.issubset(df.columns), \
        f'Missing expected columns in {Path(path).name}'
    before = len(df)
    df = df.dropna(subset=['premise', 'hypothesis', 'label']).reset_index(drop=True)
    if before != len(df):
        print(f'  Dropped {before - len(df)} rows with NaN')
    df['label'] = df['label'].astype(int)
    print(f'{Path(path).name}: {len(df)} pairs')
    print(df['label'].value_counts().sort_index())
    print()
    return df

# Load data using paths from configuration cell
train_df = load_nli_csv(TRAIN_CSV)
dev_df   = load_nli_csv(DEV_CSV)
trial_df = load_nli_csv(TRIAL_CSV)

train.csv: 24432 pairs
label
0    11784
1    12648
Name: count, dtype: int64

dev.csv: 6736 pairs
label
0    3258
1    3478
Name: count, dtype: int64

NLI_trial.csv: 50 pairs
label
0    27
1    23
Name: count, dtype: int64



## 3. Tokenisation & Vocabulary

**Note:** Using modular utilities from the `util` package:
- `tokenise()` from `util/tokenization.py`
- `Vocabulary` class from `util/vocabulary.py`

In [5]:
# ── Import tokenization from util package ────────────────────────────────────
from util.tokenization import tokenise

# tokenise all splits
train_prem_tokens = [tokenise(s) for s in train_df['premise']]
train_hyp_tokens  = [tokenise(s) for s in train_df['hypothesis']]
dev_prem_tokens   = [tokenise(s) for s in dev_df['premise']]
dev_hyp_tokens    = [tokenise(s) for s in dev_df['hypothesis']]

print('Sample premise:   ', train_df['premise'][0])
print('Tokenised:        ', train_prem_tokens[0])

Sample premise:    yeah i don't know cut California in half or something
Tokenised:         ['yeah', 'i', "don't", 'know', 'cut', 'california', 'in', 'half', 'or', 'something']


In [6]:
# ── Import Vocabulary from util package ──────────────────────────────────────
from util.vocabulary import Vocabulary

# Build vocabulary from training set only — no dev/test leakage
vocab = Vocabulary(min_freq=MIN_FREQ)
vocab.build(train_prem_tokens + train_hyp_tokens)

Vocabulary size: 22336 tokens (min_freq=2)


## 4. GloVe Embeddings

GloVe (Pennington et al., 2014) provides static 300-dimensional word vectors.
The embedding matrix is frozen during training — weights are not updated.
Words not found in GloVe are randomly initialised from U(-0.1, 0.1).
The <UNK> token is set to the mean of all GloVe vectors.

In [7]:
def load_glove(glove_path, vocab, embed_dim=EMBED_DIM):
    """Build embedding matrix [vocab_size, embed_dim] from GloVe file."""
    vocab_size = len(vocab)
    matrix = np.random.uniform(-0.1, 0.1, (vocab_size, embed_dim))
    matrix[0] = np.zeros(embed_dim)  # PAD = zeros

    glove_vectors = {}
    print('Reading GloVe file...')
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            glove_vectors[parts[0]] = np.array(parts[1:], dtype=np.float32)
    print(f'GloVe loaded: {len(glove_vectors)} vectors')

    found = 0
    for word, idx in vocab.word2idx.items():
        if word in glove_vectors:
            matrix[idx] = glove_vectors[word]
            found += 1
    matrix[1] = np.mean(list(glove_vectors.values()), axis=0)  # UNK = mean

    print(f'Words found in GloVe:     {found}')
    print(f'Words not found (random): {vocab_size - found}')
    print(f'Embedding matrix shape:   {matrix.shape}')
    return matrix

def build_glove_layer(glove_matrix):
    """Wrap GloVe matrix in a frozen PyTorch Embedding layer."""
    vocab_size, embed_dim = glove_matrix.shape
    layer = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
    layer.weight = nn.Parameter(
        torch.tensor(glove_matrix, dtype=torch.float32),
        requires_grad=False
    )
    print(f'GloVe layer: {vocab_size} x {embed_dim}, frozen=True')
    return layer

# Load GloVe using path from configuration cell
glove_matrix = load_glove(GLOVE_FILE, vocab)
glove_layer  = build_glove_layer(glove_matrix)

Reading GloVe file...
GloVe loaded: 400000 vectors
Words found in GloVe:     20866
Words not found (random): 1470
Embedding matrix shape:   (22336, 300)
GloVe layer: 22336 x 300, frozen=True


## 5. Character CNN

Character-level CNN (Kim et al., 2016) produces 100-dimensional vectors per word.
Uses three parallel 1D convolutions with kernel sizes 2, 3, 4 followed by max pooling.
This handles out-of-vocabulary words that GloVe misses (contractions, rare words).

In [8]:
def build_char_vocab(token_lists):
    """Build character vocabulary from training tokens. 0=PAD, 1=UNK."""
    char2idx = {PAD_CHAR: 0, UNK_CHAR: 1}
    for tokens in token_lists:
        for token in tokens:
            for char in token:
                if char not in char2idx:
                    char2idx[char] = len(char2idx)
    print(f'Character vocabulary size: {len(char2idx)}')
    return char2idx

char2idx = build_char_vocab(train_prem_tokens + train_hyp_tokens)

def words_to_char_tensor(token_list, char2idx,
                          max_word_len=MAX_WORD_LEN, max_seq_len=MAX_LEN):
    """Convert token list to character index tensor [max_seq_len, max_word_len]."""
    pad = char2idx[PAD_CHAR]
    unk = char2idx[UNK_CHAR]
    result = []
    for i in range(max_seq_len):
        if i < len(token_list):
            word = token_list[i][:max_word_len]
            ids  = [char2idx.get(c, unk) for c in word]
            ids  = ids + [pad] * (max_word_len - len(ids))
        else:
            ids = [pad] * max_word_len
        result.append(ids)
    return torch.tensor(result, dtype=torch.long)

class CharCNN(nn.Module):
    """
    Character-level CNN producing 100d vector per word.
    Three parallel convolutions (kernels 2,3,4) + max pooling + concat.
    Reference: Kim et al. (2016), Character-Aware Neural Language Models.
    """
    def __init__(self, char_vocab_size,
                 char_embed_dim=CHAR_EMBED_DIM, out_dim=CHAR_OUT_DIM):
        super().__init__()
        self.char_embedding = nn.Embedding(
            char_vocab_size, char_embed_dim, padding_idx=0)
        self.num_filters = out_dim // 3
        self.conv2 = nn.Conv1d(char_embed_dim, self.num_filters,     kernel_size=2)
        self.conv3 = nn.Conv1d(char_embed_dim, self.num_filters,     kernel_size=3)
        self.conv4 = nn.Conv1d(char_embed_dim, self.num_filters + 1, kernel_size=4)
        self.dropout = nn.Dropout(0.3)

    def forward(self, char_ids):
        # char_ids: [batch, seq_len, max_word_len]
        batch, seq_len, max_word_len = char_ids.shape
        char_ids = char_ids.view(batch * seq_len, max_word_len)
        x = self.char_embedding(char_ids).transpose(1, 2)
        x2 = F.max_pool1d(F.relu(self.conv2(x)), self.conv2(x).shape[-1]).squeeze(-1)
        x3 = F.max_pool1d(F.relu(self.conv3(x)), self.conv3(x).shape[-1]).squeeze(-1)
        x4 = F.max_pool1d(F.relu(self.conv4(x)), self.conv4(x).shape[-1]).squeeze(-1)
        out = self.dropout(torch.cat([x2, x3, x4], dim=-1))
        return out.view(batch, seq_len, -1)  # [batch, seq_len, 100]

char_cnn = CharCNN(char_vocab_size=len(char2idx))
print(f'CharCNN output dim: {CHAR_OUT_DIM}')

Character vocabulary size: 29
CharCNN output dim: 100


## 6. POS Embeddings

Learned 50-dimensional embeddings for Universal POS tags (spaCy).
Unlike GloVe, these are updated during training.
Provides explicit syntactic signal complementary to ELMo's implicit encoding.

In [9]:
def build_pos_vocab(token_lists):
    """Collect POS tags from training data via spaCy. 0=PAD, 1=UNK."""
    pos2idx = {PAD_POS: 0, UNK_POS: 1}
    for tokens in token_lists:
        doc = nlp(' '.join(tokens))
        for token in doc:
            if token.pos_ not in pos2idx:
                pos2idx[token.pos_] = len(pos2idx)
    print(f'POS vocabulary: {len(pos2idx)} tags')
    print(f'Tags: {list(pos2idx.keys())}')
    return pos2idx

def get_pos_ids(token_list, pos2idx, max_len=MAX_LEN):
    """Return POS tag indices for a token list, padded to max_len."""
    unk = pos2idx[UNK_POS]
    pad = pos2idx[PAD_POS]
    doc = nlp(' '.join(token_list))
    ids = [pos2idx.get(t.pos_, unk) for t in doc]
    ids = ids[:max_len] + [pad] * (max_len - len(ids))
    return ids

class POSEmbedding(nn.Module):
    """Learned POS tag embedding layer. Updated during training."""
    def __init__(self, pos_vocab_size, pos_embed_dim=POS_EMBED_DIM):
        super().__init__()
        self.embedding = nn.Embedding(
            pos_vocab_size, pos_embed_dim, padding_idx=0)

    def forward(self, pos_ids):
        return self.embedding(pos_ids)  # [batch, seq_len, 50]

pos2idx       = build_pos_vocab(train_prem_tokens + train_hyp_tokens)
pos_embedding = POSEmbedding(pos_vocab_size=len(pos2idx))

POS vocabulary: 19 tags
Tags: ['<PAD>', '<UNK>', 'INTJ', 'PRON', 'AUX', 'PART', 'VERB', 'PROPN', 'ADP', 'NOUN', 'CCONJ', 'ADJ', 'DET', 'SCONJ', 'ADV', 'NUM', 'X', 'PUNCT', 'SYM']


## 7. Negation Flags

Rule-based 2-dimensional binary feature per token:
- dim 0: is this token a negation word?
- dim 1: is this token in the scope of a negation? (head verb of neg dependency)

Motivation: NLI models frequently fail on negation. GloVe has no context awareness
for negation scope. ELMo captures it implicitly but this explicit signal helps the
negation gate in the attention module.

In [10]:
def get_negation_flags(token_list, max_len=MAX_LEN):
    """
    Returns [max_len, 2] float32 tensor.
    dim 0: is_neg  — token is a negation word or has neg dependency
    dim 1: in_scope — token is the head verb of a negation
    """
    doc = nlp(' '.join(token_list[:max_len]))
    negated_heads = {t.head.i for t in doc if t.dep_ == 'neg'}
    flags = []
    for t in doc:
        is_neg   = 1.0 if t.text in NEGATION_WORDS or t.dep_ == 'neg' else 0.0
        in_scope = 1.0 if t.i in negated_heads else 0.0
        flags.append([is_neg, in_scope])
    flags = flags[:max_len] + [[0.0, 0.0]] * (max_len - len(flags))
    return torch.tensor(flags, dtype=torch.float32)

## 8. ELMo Pre-computation

ELMo (Peters et al., 2018) produces contextual 1024-dimensional embeddings.
Unlike GloVe, ELMo uses a 2-layer BiLSTM over the full sentence, producing
different vectors for the same word in different contexts.

**Implementation note:** AllenNLP (the original ELMo library) requires Python ≤ 3.10
and PyTorch < 1.13. We use a Python 3.10 venv to run pre-computation once,
save vectors to disk, then load them in the main kernel for training.
This is consistent with standard practice (Peters et al. recommend pre-computing
ELMo representations for downstream tasks).

In [11]:
# ── Write ELMo pre-computation script ────────────────────────────────────────
# This script should be run in a Python 3.10 environment with AllenNLP installed

elmo_script = f'''
import numpy as np
import pandas as pd
import re
import torch
from pathlib import Path
from allennlp.modules.elmo import Elmo, batch_to_ids

# Use paths from configuration
TRAIN_PATH   = Path("{TRAIN_CSV}")
DEV_PATH     = Path("{DEV_CSV}")
ELMO_OPTIONS = Path("{ELMO_OPTIONS_FILE}")
ELMO_WEIGHTS = Path("{ELMO_WEIGHTS_FILE}")
OUTPUT_DIR   = Path("{OUTPUT_DIR}")
MAX_LEN      = 64
BATCH_SIZE   = 64

OUTPUT_DIR.mkdir(exist_ok=True)

print("Loading ELMo (Peters et al., 2018) via AllenNLP...")
elmo   = Elmo(str(ELMO_OPTIONS), str(ELMO_WEIGHTS), num_output_representations=1, dropout=0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
elmo   = elmo.to(device).eval()
print(f"ELMo loaded on {{device}}")

def tokenise(text):
    return re.findall(r"[a-z]+(?:'[a-z]+)*", text.lower().strip())

def load_csv(path):
    df = pd.read_csv(path)
    return df["premise"].tolist(), df["hypothesis"].tolist()

def compute_elmo(sentences, name):
    print(f"Computing ELMo for {{len(sentences)}} {{name}} sentences...")
    token_lists = [tokenise(s) for s in sentences]
    token_lists = [t[:MAX_LEN] if len(t) > 0 else ["unk"] for t in token_lists]
    all_embeddings = np.zeros((len(token_lists), MAX_LEN, 1024), dtype=np.float32)
    for i in range(0, len(token_lists), BATCH_SIZE):
        batch    = token_lists[i:i+BATCH_SIZE]
        char_ids = batch_to_ids(batch).to(device)
        with torch.no_grad():
            output = elmo(char_ids)
        emb = output["elmo_representations"][0].cpu().numpy()
        for j in range(len(batch)):
            seq_len = min(len(batch[j]), MAX_LEN)
            all_embeddings[i+j, :seq_len, :] = emb[j, :seq_len, :]
        if (i // BATCH_SIZE) % 10 == 0:
            print(f"  {{min(i+BATCH_SIZE, len(token_lists))}}/{{len(token_lists)}}")
    print(f"  Done. Shape: {{all_embeddings.shape}}")
    return all_embeddings

train_prem, train_hyp = load_csv(TRAIN_PATH)
np.save(OUTPUT_DIR / "elmo_train_prem.npy", compute_elmo(train_prem, "train premises"))
np.save(OUTPUT_DIR / "elmo_train_hyp.npy",  compute_elmo(train_hyp,  "train hypotheses"))
print("Train saved.")

dev_prem, dev_hyp = load_csv(DEV_PATH)
np.save(OUTPUT_DIR / "elmo_dev_prem.npy", compute_elmo(dev_prem, "dev premises"))
np.save(OUTPUT_DIR / "elmo_dev_hyp.npy",  compute_elmo(dev_hyp,  "dev hypotheses"))
print("Dev saved.")
print("ALL DONE.")
'''

# Write script to file using path from configuration cell
with open(ELMO_PRECOMPUTE_SCRIPT, 'w') as f:
    f.write(elmo_script)
print(f'elmo_precompute.py written to: {ELMO_PRECOMPUTE_SCRIPT}')
print('\nTo run this script:')
print('1. Create a Python 3.10 virtual environment')
print('2. Install: pip install allennlp allennlp-models "numpy<2.0" torch')
print(f'3. Run: python {ELMO_PRECOMPUTE_SCRIPT.name}')

elmo_precompute.py written to: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/elmo_precompute.py

To run this script:
1. Create a Python 3.10 virtual environment
2. Install: pip install allennlp allennlp-models "numpy<2.0" torch
3. Run: python elmo_precompute.py


In [25]:
# ── Run ELMo pre-computation (OPTIONAL) ───────────────────────────────────────
# Skip this cell if you already have pre-computed embeddings or want to skip ELMo

# To run this, you need a Python 3.10 environment with AllenNLP
# Uncomment the following line to run:
# !python {ELMO_PRECOMPUTE_SCRIPT}

print("Note: ELMo pre-computation requires Python 3.10 with AllenNLP")
print("If you haven't set this up yet, you can skip ELMo for now.")
print(f"To run later: create a Python 3.10 venv and run: python {ELMO_PRECOMPUTE_SCRIPT.name}")

Note: ELMo pre-computation requires Python 3.10 with AllenNLP
If you haven't set this up yet, you can skip ELMo for now.
To run later: create a Python 3.10 venv and run: python elmo_precompute.py


In [26]:
# ── Load pre-computed ELMo arrays ────────────────────────────────────────────
import numpy as np
from pathlib import Path

# Check if pre-computed embeddings exist (using paths from configuration cell)
if all(p.exists() for p in [ELMO_TRAIN_PREM, ELMO_TRAIN_HYP, ELMO_DEV_PREM, ELMO_DEV_HYP]):
    print("Loading pre-computed ELMo embeddings...")
    elmo_train_prem = np.load(ELMO_TRAIN_PREM)
    elmo_train_hyp  = np.load(ELMO_TRAIN_HYP)
    elmo_dev_prem   = np.load(ELMO_DEV_PREM)
    elmo_dev_hyp    = np.load(ELMO_DEV_HYP)
    
    print(f'elmo_train_prem: {elmo_train_prem.shape}')
    print(f'elmo_train_hyp:  {elmo_train_hyp.shape}')
    print(f'elmo_dev_prem:   {elmo_dev_prem.shape}')
    print(f'elmo_dev_hyp:    {elmo_dev_hyp.shape}')
else:
    print("Pre-computed ELMo embeddings not found.")
    print(f"Expected location: {OUTPUT_DIR}")
    print("\nOptions:")
    print(f"1. Run {ELMO_PRECOMPUTE_SCRIPT.name} in a Python 3.10 environment")
    print("2. Create dummy embeddings for testing (see below)")
    print("\nCreating dummy zero embeddings for testing...")
    
    # Create dummy embeddings with correct shape
    num_train = len(train_df)
    num_dev = len(dev_df)
    
    elmo_train_prem = np.zeros((num_train, MAX_LEN, 1024), dtype=np.float32)
    elmo_train_hyp  = np.zeros((num_train, MAX_LEN, 1024), dtype=np.float32)
    elmo_dev_prem   = np.zeros((num_dev, MAX_LEN, 1024), dtype=np.float32)
    elmo_dev_hyp    = np.zeros((num_dev, MAX_LEN, 1024), dtype=np.float32)
    
    print(f'Using dummy zero embeddings:')
    print(f'elmo_train_prem: {elmo_train_prem.shape}')
    print(f'elmo_train_hyp:  {elmo_train_hyp.shape}')
    print(f'elmo_dev_prem:   {elmo_dev_prem.shape}')
    print(f'elmo_dev_hyp:    {elmo_dev_hyp.shape}')
    print('\nWARNING: Using dummy embeddings. Model will not have ELMo features!')

Pre-computed ELMo embeddings not found.
Expected location: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/output

Options:
1. Run elmo_precompute.py in a Python 3.10 environment
2. Create dummy embeddings for testing (see below)

Creating dummy zero embeddings for testing...
Using dummy zero embeddings:
elmo_train_prem: (24432, 64, 1024)
elmo_train_hyp:  (24432, 64, 1024)
elmo_dev_prem:   (6736, 64, 1024)
elmo_dev_hyp:    (6736, 64, 1024)



## 9. Input Embedding Module

Combines all five components into a single [batch, seq_len, 1476] tensor.
This is the interface between the embedding pipeline and ResESIM.

In [27]:
class InputEmbeddingModule(nn.Module):
    """
    Combines GloVe + ELMo + CharCNN + POS + Negation flags.
    Output: [batch, seq_len, 1476]

    Component breakdown:
      GloVe  [300d] — Pennington et al. (2014), frozen
      ELMo  [1024d] — Peters et al. (2018), pre-computed frozen
      CharCNN[100d] — Kim et al. (2016), learned
      POS    [ 50d] — learned
      Negation[ 2d] — rule-based spaCy dependency parse
    """
    def __init__(self, glove_layer, char_cnn, pos_embedding, dropout=0.5):
        super().__init__()
        self.glove_layer   = glove_layer
        self.char_cnn      = char_cnn
        self.pos_embedding = pos_embedding
        self.dropout       = nn.Dropout(dropout)
        self.output_dim    = 300 + 1024 + 100 + 50 + 2  # 1476

    def forward(self, token_ids, char_ids, pos_ids, neg_flags, elmo_vecs):
        glove_out = self.glove_layer(token_ids)    # [batch, seq_len, 300]
        char_out  = self.char_cnn(char_ids)        # [batch, seq_len, 100]
        pos_out   = self.pos_embedding(pos_ids)    # [batch, seq_len,  50]
        elmo_out  = elmo_vecs.float()              # [batch, seq_len,1024]
        out = torch.cat([glove_out, elmo_out, char_out, pos_out, neg_flags], dim=-1)
        return self.dropout(out)                   # [batch, seq_len,1476]

input_module = InputEmbeddingModule(
    glove_layer   = glove_layer,
    char_cnn      = char_cnn,
    pos_embedding = pos_embedding,
    dropout       = 0.5
)

print(f'InputEmbeddingModule output dim: {input_module.output_dim}')
print(f'Trainable parameters: {sum(p.numel() for p in input_module.parameters() if p.requires_grad):,}')
print(f'Frozen parameters:    {sum(p.numel() for p in input_module.parameters() if not p.requires_grad):,}')

InputEmbeddingModule output dim: 1476
Trainable parameters: 10,950
Frozen parameters:    6,700,800


## 10. Dataset & DataLoaders

**Note:** We use the `NLIDataset` class from the `util` package (see `util/dataset.py`).

The dataset returns all tensors needed by InputEmbeddingModule and OracleNet in one batch dict.
No on-the-fly ELMo computation — vectors are loaded from pre-computed arrays.

### What NLIDataset stores:
- Pre-tokenized text (computed once at initialization)
- Pre-computed ELMo arrays (⚠️ ~5GB for training data)
- References to vocabularies

### What gets computed on-the-fly (every `__getitem__`):
- Token IDs (vocabulary lookup)
- Character tensors
- POS tags (spaCy processing)
- Negation flags (spaCy dependency parsing)
- Padding masks

In [28]:
# ── Import NLIDataset from util package ──────────────────────────────────────
from util.dataset import NLIDataset

# Create datasets using the modular NLIDataset class
train_dataset = NLIDataset(train_df, vocab, char2idx, pos2idx, elmo_train_prem, elmo_train_hyp)
dev_dataset   = NLIDataset(dev_df,   vocab, char2idx, pos2idx, elmo_dev_prem,   elmo_dev_hyp)

# Create DataLoaders
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
dev_loader    = DataLoader(dev_dataset,   batch_size=32, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Dev:   {len(dev_dataset)} samples, {len(dev_loader)} batches')

Train: 24432 samples, 764 batches
Dev:   6736 samples, 211 batches


In [29]:
# ── End-to-end sanity check ───────────────────────────────────────────────────
batch = next(iter(train_loader))

input_module.eval()
with torch.no_grad():
    prem_out = input_module(
        batch['premise_ids'], batch['premise_char'],
        batch['premise_pos'], batch['premise_neg'], batch['premise_elmo']
    )
    hyp_out = input_module(
        batch['hypothesis_ids'], batch['hypothesis_char'],
        batch['hypothesis_pos'], batch['hypothesis_neg'], batch['hypothesis_elmo']
    )

print(f'Premise output:    {prem_out.shape}  — expected [32, 64, 1476]')
print(f'Hypothesis output: {hyp_out.shape}  — expected [32, 64, 1476]')
print(f'Premise mask:      {batch["premise_mask"].shape}')
print(f'Labels:            {batch["label"].shape}')
print(f'Sample labels:     {batch["label"][:8]}')

# lengths for OracleNet
prem_lens = batch['premise_mask'].sum(dim=1)
hyp_lens  = batch['hypothesis_mask'].sum(dim=1)
print(f'\nPremise lengths (first 8): {prem_lens[:8]}')
print(f'Hypothesis lengths (first 8): {hyp_lens[:8]}')
print('\nOracleNet call:')
print('  logits = oracle_net(prem_out, hyp_out, prem_lens, hyp_lens)')
print('  input_dim=1476, num_classes=2')

UnboundLocalError: Caught UnboundLocalError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/venv/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/venv/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/venv/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 54, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/util/dataset.py", line 105, in __getitem__
    'premise_char': words_to_char_tensor(prem_tok, self.char2idx, 20, self.max_len),
UnboundLocalError: local variable 'words_to_char_tensor

## 11. Save to Drive

Save all artefacts to Google Drive for session persistence.
After a session reset, reload using the commented code in Section 8.

In [ ]:
# ── Save embeddings locally ──────────────────────────────────────────────────
import os
from pathlib import Path

OUTPUT_DIR.mkdir(exist_ok=True)

# Bundle 1 — vocab and GloVe matrix (small, ~80MB)
torch.save({
    'vocab':        vocab,
    'char2idx':     char2idx,
    'pos2idx':      pos2idx,
    'glove_matrix': glove_matrix,
}, VOCAB_GLOVE_SAVE)
print(f'vocab_and_glove.pt saved to: {VOCAB_GLOVE_SAVE}')

# Bundle 2 — ELMo arrays compressed (~3.5GB)
# Only save if we have real ELMo embeddings (not dummy zeros)
if not np.all(elmo_train_prem == 0):
    np.savez_compressed(ELMO_ARRAYS_SAVE,
        train_prem = elmo_train_prem,
        train_hyp  = elmo_train_hyp,
        dev_prem   = elmo_dev_prem,
        dev_hyp    = elmo_dev_hyp
    )
    print(f'elmo_arrays.npz saved to: {ELMO_ARRAYS_SAVE}')
else:
    print('Skipping ELMo save (using dummy embeddings)')

# Show file sizes
print('\nSaved files:')
for f in OUTPUT_DIR.glob('*.pt'):
    size = f.stat().st_size
    print(f'  {f.name}: {size/1e6:.2f} MB')
for f in OUTPUT_DIR.glob('*.npz'):
    size = f.stat().st_size
    print(f'  {f.name}: {size/1e9:.2f} GB')

print(f'\nAll artifacts saved to: {OUTPUT_DIR}')

In [ ]:
# Cell 0: Configuration - File Paths
from pathlib import Path

# ── Root Directories ──────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()                    # Current working directory
DATA_DIR = PROJECT_ROOT                      # Where CSV files are located
BIN_DIR = PROJECT_ROOT / 'bin'               # Where downloaded binaries go
OUTPUT_DIR = PROJECT_ROOT / 'output'         # Where outputs are saved

# ── Data Files (CSV) ──────────────────────────────────────────────────────────
TRAIN_CSV = DATA_DIR / 'train.csv'
DEV_CSV = DATA_DIR / 'dev.csv'
TRIAL_CSV = DATA_DIR / 'NLI_trial.csv'

# ── Binary Directories ────────────────────────────────────────────────────────
GLOVE_DIR = BIN_DIR / 'glove'                # GloVe embeddings
ELMO_DIR = BIN_DIR / 'elmo'                  # ELMo model files

# ── Specific Binary Files ─────────────────────────────────────────────────────
GLOVE_FILE = GLOVE_DIR / 'glove.6B.300d.txt'
ELMO_OPTIONS_FILE = ELMO_DIR / 'elmo_2x4096_512_2048cnn_2xhighway_options.json'
ELMO_WEIGHTS_FILE = ELMO_DIR / 'elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5'

# ── Pre-computed ELMo Embeddings ──────────────────────────────────────────────
ELMO_TRAIN_PREM = OUTPUT_DIR / 'elmo_train_prem.npy'
ELMO_TRAIN_HYP = OUTPUT_DIR / 'elmo_train_hyp.npy'
ELMO_DEV_PREM = OUTPUT_DIR / 'elmo_dev_prem.npy'
ELMO_DEV_HYP = OUTPUT_DIR / 'elmo_dev_hyp.npy'

# ── Saved Artifacts ───────────────────────────────────────────────────────────
VOCAB_GLOVE_SAVE = OUTPUT_DIR / 'vocab_and_glove.pt'
ELMO_ARRAYS_SAVE = OUTPUT_DIR / 'elmo_arrays.npz'

# ── ELMo Pre-computation Script ───────────────────────────────────────────────
ELMO_PRECOMPUTE_SCRIPT = PROJECT_ROOT / 'elmo_precompute.py'

# ── Print Configuration ───────────────────────────────────────────────────────
print("=" * 80)
print("FILE PATH CONFIGURATION")
print("=" * 80)
print(f"Project Root:     {PROJECT_ROOT}")
print(f"Data Directory:   {DATA_DIR}")
print(f"Binary Directory: {BIN_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print()
print("Data Files:")
print(f"  Train CSV:      {TRAIN_CSV}")
print(f"  Dev CSV:        {DEV_CSV}")
print(f"  Trial CSV:      {TRIAL_CSV}")
print()
print("Binary Files:")
print(f"  GloVe:          {GLOVE_FILE}")
print(f"  ELMo Options:   {ELMO_OPTIONS_FILE}")
print(f"  ELMo Weights:   {ELMO_WEIGHTS_FILE}")
print()
print("Output Files:")
print(f"  Vocab & GloVe:  {VOCAB_GLOVE_SAVE}")
print(f"  ELMo Arrays:    {ELMO_ARRAYS_SAVE}")
print("=" * 80)

FILE PATH CONFIGURATION
Project Root:     /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW
Data Directory:   /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW
Binary Directory: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin
Output Directory: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/output

Data Files:
  Train CSV:      /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/train.csv
  Dev CSV:        /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/dev.csv
  Trial CSV:      /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/NLI_trial.csv

Binary Files:
  GloVe:          /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/glove/glove.6B.300d.txt
  ELMo Options:   /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo/elmo_2x4096_512_2048cnn_2xhighway_options.json
  ELMo Weights:   /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5

Output Files:
  Vocab & GloVe:  /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/output/vocab_and_glove.p

In [ ]:
# Cell 1: Environment Setup - Download binaries
import sys
import os
from pathlib import Path

# Add current directory to path to import elmo package
sys.path.insert(0, str(Path.cwd()))

from elmo.downloader import download_and_extract_glove_6B, elmo_downloader

# ── Setup directories ──────────────────────────────────────────────────────────
BIN_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# ── GloVe weights (Pennington et al., 2014) ───────────────────────────────────
print("Downloading GloVe embeddings...")
glove_dir = download_and_extract_glove_6B(download_folder=str(GLOVE_DIR))
print(f"GloVe downloaded to: {glove_dir}")

# ── ELMo weights (Peters et al., 2018) — original AllenNLP release ───────────
print("\nDownloading ELMo model files...")
elmo_dir = elmo_downloader(download_folder=str(ELMO_DIR))
print(f"ELMo downloaded to: {elmo_dir}")

print('\nEnvironment setup complete.')
print(f'Binary files location: {BIN_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

Zip file already exists at /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/glove/glove.6B.zip
Skipping download.
Extracting files to /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/glove
Extraction complete. GloVe files available in /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/glove
GloVe downloaded to: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/glove

ELMo options file already exists at /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo/elmo_2x4096_512_2048cnn_2xhighway_options.json
Skipping download.
ELMo weights file already exists at /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5
Skipping download.
ELMo model files available in /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo
ELMo downloaded to: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin/elmo

Environment setup complete.
Binary files location: /Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/bin
Output directory: /Users/vpremakant

In [ ]:
# Cell 2: Main kernel imports and constants
import re
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import spacy
nlp = spacy.load('en_core_web_sm')

# ── Constants ─────────────────────────────────────────────────────────────────
PAD_TOKEN    = '<PAD>'
UNK_TOKEN    = '<UNK>'
PAD_CHAR     = '<PAD>'
UNK_CHAR     = '<UNK>'
PAD_POS      = '<PAD>'
UNK_POS      = '<UNK>'
MAX_LEN      = 64
MAX_WORD_LEN = 20
MIN_FREQ     = 2
EMBED_DIM    = 300
CHAR_EMBED_DIM = 30
CHAR_OUT_DIM   = 100
POS_EMBED_DIM  = 50
NUM_CLASSES    = 2

NEGATION_WORDS = {
    'not', 'no', 'never', 'neither', 'nor', 'nobody', 'nothing',
    'nowhere', 'hardly', 'barely', 'scarcely', "n't", 'cannot'
}

print('Imports and constants OK')
print(f'Using paths from configuration cell above')

Imports and constants OK
Using paths from configuration cell above


In [13]:
# Cell 3: Data Loading
def load_nli_csv(path):
    """Load NLI CSV with columns: premise, hypothesis, label."""
    df = pd.read_csv(Path(path))
    assert {'premise', 'hypothesis', 'label'}.issubset(df.columns), \
        f'Missing expected columns in {Path(path).name}'
    before = len(df)
    df = df.dropna(subset=['premise', 'hypothesis', 'label']).reset_index(drop=True)
    if before != len(df):
        print(f'  Dropped {before - len(df)} rows with NaN')
    df['label'] = df['label'].astype(int)
    print(f'{Path(path).name}: {len(df)} pairs')
    print(df['label'].value_counts().sort_index())
    print()
    return df

# Load data using paths from configuration cell
train_df = load_nli_csv(TRAIN_CSV)
dev_df   = load_nli_csv(DEV_CSV)
trial_df = load_nli_csv(TRIAL_CSV)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/vpremakantha/Documents/UOM/Y3/NLU/NLU-CW/train.csv'